# Convert Manga109 to Hugging Face Dataset

## Environment Variables

In [67]:
# The Hugging Face token with read access to the Manga109 dataset.
HF_TOKEN_MANGA109_READ = ""

# The zip file of the Manga109 dataset can be downloaded from the Hugging Face Hub. 
# You can find it at: https://huggingface.co/datasets/hal-utokyo/Manga109/tree/main
MANGA109_ZIP_FILENAME = "Manga109_released_2023_12_07.zip"

# The Hugging Face token with write access to the repository where the converted dataset will be uploaded.
HF_TOKEN = ""
HF_UPLOAD_REPO = "" # eg. <your-username>/manga109

In [ ]:
!pip install manga109api

from pprint import pprint
import pandas as pd
from datasets.load import load_dataset

## Download Manga109 dataset

> Required to `HF_TOKEN_MANGA109_READ` environment variable with a Hugging Face token that has read access to the `manga109` dataset.


In [ ]:
from huggingface_hub import hf_hub_download
import zipfile
import os
import shutil


# Download manga109 dataset from Hugging Face Hub
def download_manga109_dataset():
    zip_ds_path = hf_hub_download(
        repo_id="hal-utokyo/Manga109",
        filename=MANGA109_ZIP_FILENAME,
        repo_type="dataset",
        token=HF_TOKEN_MANGA109_READ,
    )
    print(zip_ds_path)

    # Unzip the downloaded file
    unzipped_dir = f".tmp/manga109_dataset"
    with zipfile.ZipFile(zip_ds_path, "r") as zip_ref:
        zip_ref.extractall(unzipped_dir)

    data_dir = f"{unzipped_dir}/{MANGA109_ZIP_FILENAME[:-4]}"

    # Move the unzipped directory to permanent location
    final_dir = f"datasets/Manga109"
    if os.path.exists(final_dir):
        shutil.rmtree(final_dir)
    shutil.move(data_dir, final_dir)

    return final_dir


manga109_ds_root_dir = download_manga109_dataset()

pprint(f"Downloaded and unzipped Manga109 dataset to: {manga109_ds_root_dir}")

## Build Hugging Face Dataset

Edit the `use_image_path` argument to `False` if you want to save the images directly in the Hugging Face Dataset instead of saving the image paths. Note that saving images directly in the dataset will significantly increase the dataset size.


In [ ]:
import manga109api
import datasets

parser = manga109api.Parser(root_dir=manga109_ds_root_dir)


# Format to convert bounding box coordinates to [xmin, ymin, xmax, ymax]
def convert_to_bbox_array(xmin: int, ymin: int, xmax: int, ymax: int):
    return [xmin, ymin, xmax, ymax]


def ensure_list(annotations):
    if annotations is None:
        print("No annotations found, returning empty list.")
        return []
    if isinstance(annotations, list):
        return annotations
    raise ValueError(
        f"Expected annotations to be a list or None, but got {type(annotations)}"
    )


def convert_basic_annotations_dataset_format(annotations):
    dataset_format = []
    for annotation in ensure_list(annotations):
        if not isinstance(annotation, dict):
            raise ValueError(
                f"Expected annotation to be a dict, but got {type(annotation)}"
            )

        dataset_format.append(
            {
                "bbox": convert_to_bbox_array(
                    int(annotation["@xmin"]),
                    int(annotation["@ymin"]),
                    int(annotation["@xmax"]),
                    int(annotation["@ymax"]),
                ),
            }
        )
    return dataset_format


def convert_text_annotations_dataset_format(annotations):
    dataset_format = convert_basic_annotations_dataset_format(annotations)
    for i, annotation in enumerate(ensure_list(annotations)):
        dataset_format[i]["text"] = (
            annotation.get("#text", "") if isinstance(annotation, dict) else ""
        )
    return dataset_format


# Build dataset as list of dicts
def build_dataset(use_image_path=True):
    dataset = []
    for book_name in parser.books:
        annotation = parser.get_annotation(book_name)
        for page in annotation["page"]:
            image_path = parser.img_path(book=book_name, index=page["@index"])

            dataset.append(
                {
                    "book": book_name,
                    "annotations": {
                        "frame": convert_basic_annotations_dataset_format(
                            page.get("frame", [])
                        ),
                        "face": convert_basic_annotations_dataset_format(
                            page.get("face", [])
                        ),
                        "body": convert_basic_annotations_dataset_format(
                            page.get("body", [])
                        ),
                        "text": convert_text_annotations_dataset_format(
                            page.get("text", [])
                        ),
                    },
                    "image": {
                        "path": image_path if use_image_path else None,
                        "bytes": None if use_image_path else image_path,
                    },
                }
            )

    # Build features
    #! Make sure to match the structure of the dataset entries when defining features
    basic_annotation_features = [
        {
            "bbox": [datasets.Value("int32")],
        }
    ]
    text_annotation_features = [
        {
            "bbox": [datasets.Value("int32")],
            "text": datasets.Value("string"),
        }
    ]
    features = Features(
        {
            "book": datasets.Value("string"),
            "annotations": {
                "frame": basic_annotation_features,
                "face": basic_annotation_features,
                "body": basic_annotation_features,
                "text": text_annotation_features,
            },
            "image": {
                "path": datasets.Value("string"),
                "bytes": datasets.Image(),
            },
        }
    )
    
    return dataset, features

# Build dataset as Hugging Face Dataset
def build_hf_dataset(use_image_path=True):
    dataset, features = build_dataset(use_image_path=use_image_path)
    return datasets.Dataset.from_list(dataset, features=features)

ds = build_hf_dataset(use_image_path=True)

## Upload to Hugging Face Hub

> Required to `HF_TOKEN` environment variable with a Hugging Face token that has write access to the `HF_UPLOAD_REPO` repository.

In [68]:
ds.push_to_hub(HF_UPLOAD_REPO, token=HF_TOKEN)